In [11]:
# !pip install fusion_solar_py -q

In [31]:
from fusion_solar_py.client import FusionSolarClient, logged_in
from fusion_solar_py.exceptions import FusionSolarException
from enum import Enum


In [12]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")

In [33]:
class FusionSolarClientExtended(FusionSolarClient):
    """Extension of FusionSolarClient with additional functionality."""

    class BatteryWorkingMode(Enum):
        MAXIMUM_SELF_CONSUMPTION = 2
        FULLY_FEED_TO_GRID = 4

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    @logged_in
    def set_battery_working_mode(self, battery_id, mode: BatteryWorkingMode):
        if not isinstance(mode, self.BatteryWorkingMode):
            raise ValueError(
                f"Invalid mode: {mode}. Expected one of {[e.name for e in self.BatteryWorkingMode]}"
            )

        url = f"https://{self._huawei_subdomain}.fusionsolar.huawei.com/rest/pvms/web/device/v1/deviceExt/set-config-signals"
        data = {
            "dn": battery_id,
            "changeValues": f'[{{"id":"230320241","value":"{mode.value}"}}]',
        }

        response = self._session.post(url, data=data)
        response.raise_for_status()

        try:
            response_json = response.json()
            for data in response_json['data']:
                if data['code'] != 0:
                    raise FusionSolarException(
                        f"Failed to set mode for battery {data['dn']}"
                    )
        except ValueError:
            print("Error: The response is not in JSON format.")


In [34]:
# log into the API - with proper credentials...
client = FusionSolarClientExtended(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                   huawei_subdomain="uni004eu5")
plant_id = client.get_plant_ids()[0]
battery_id = client.get_battery_ids(plant_id)[0]

In [37]:
client.set_battery_working_mode(battery_id, client.BatteryWorkingMode.MAXIMUM_SELF_CONSUMPTION)